In [1]:
import asyncio
import logging
from rose.metrics import GREATER_THAN_THRESHOLD
from rose.rl.reinforcement_learner import SequentialReinforcementLearner

from radical.asyncflow import WorkflowEngine
from radical.asyncflow import ConcurrentExecutionBackend
from concurrent.futures import ProcessPoolExecutor
from radical.asyncflow.logging import init_default_logger

logger = logging.getLogger(__name__)

In [ ]:
run_name="roserun-1"
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('hf_token')
    userdata.get('hf')
except:
    os.environ["HF_TOKEN"] = ""
    os.environ["HF_HOME"] = "/work/hdd/bdyk/apark4/huggingface"
import torch
import re
import random
import time
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from transformers.trainer_utils import get_last_checkpoint
from trl import GRPOConfig, GRPOTrainer
import trackio
import gc
trackio.init(project="huggingface", space_id="iznoanygod/trackio", name=run_name, embed=False)

* Trackio project initialized: huggingface
* Trackio metrics will be synced to Hugging Face Dataset: iznoanygod/trackio-dataset
* Found existing space: https://huggingface.co/spaces/iznoanygod/trackio
* View dashboard by going to: https://iznoanygod-trackio.hf.space/
* Created new run: roserun-1


In [3]:
engine = await ConcurrentExecutionBackend(ProcessPoolExecutor())
asyncflow = await WorkflowEngine.create(engine)
rl = SequentialReinforcementLearner(asyncflow)

In [4]:
max_seq_length = 2048
max_prompt_length = 1024
lora_rank = 16

base_model_id="meta-llama/Llama-3.1-8B-Instruct"
lora_id = "math_lora"

In [5]:
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
You must use LaTeX to format mathematical expressions, and you must use \\boxed{...} to indicate the final answer.
"""

In [6]:
def get_answer(expr: str):
    match = re.search(r"\\boxed\{(.+?)\}", expr)
    if match:
        return match.group(1).strip()
    return None

def correctness_reward_func(prompts, completions, ground_truth, **kwargs):
    rewards = []
    for prompt, completion, ground in zip(prompts, completions, ground_truth):
        c = get_answer(completion[0]["content"])
        g = get_answer(ground)
        reward = 2.0 if g == c else 0
        rewards.append(reward)
    return rewards

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

In [7]:
def to_prompt_completion(example):
    return {
        "prompt": [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': example['problem']}
            
        ],
        "ground_truth": str(example["solution"]).strip(),
    }

#dataset = load_dataset("qwedsacf/competition_math", split="train")
#mapped = dataset.map(to_prompt_completion, remove_columns=dataset.column_names)

In [8]:
def load_llama_or_latest_checkpoint(
    base_model_id: str,
    lora_id: str,
    dtype=torch.bfloat16,
    device_map="auto",
):
    """
    If `output_dir` contains checkpoints for this run, load the latest one.
    Otherwise, load the base model from Hugging Face Hub.
    """

    last_checkpoint = None

    if os.path.isdir(lora_id):
        last_checkpoint = get_last_checkpoint(lora_id)
        
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, padding_side="left", use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token
    iteration = 0
    if last_checkpoint is not None:
        print(f"Found LoRA checkpoint at: {last_checkpoint}")
        print(f"Loading base model: {base_model_id}")
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            dtype=dtype,
            device_map=device_map,
        )
        m = re.search(r"checkpoint-(\d+)", last_checkpoint)
        if m:
            iteration = int(m.group(1))
        else:
            print(f"Warning: could not parse iteration from {basename}, leaving iteration=0")
        # Attach LoRA adapter weights
        print("Applying LoRA adapter from checkpoint...")
        model = PeftModel.from_pretrained(base_model, last_checkpoint, is_trainable=True)
        loaded_from = last_checkpoint
    else:
        print(f"No checkpoint found, loading base model: {base_model_id}")
        lora_config = LoraConfig(
            r=lora_rank,
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            dtype=dtype,
            device_map=device_map,
        )
        model = get_peft_model(model, lora_config)
        loaded_from = base_model_id

    return model, tokenizer, loaded_from, iteration

In [11]:
def memory_stats():
    print("memory allocated: ", torch.cuda.memory_allocated()/1024**2)
    print("memory reserved: ", torch.cuda.memory_reserved()/1024**2)

In [13]:
async def run(rl, **kwargs):
    @rl.update_task(as_executable=False)
    async def update(*args, **kwargs) -> dict:
        #need to let the previous node get cleaned up
        time.sleep(10)
        model, tokenizer, loaded_from, iteration = load_llama_or_latest_checkpoint(
            base_model_id=base_model_id,
            lora_id=lora_id,
            dtype=torch.bfloat16,
        )
        model.print_trainable_parameters()
        logger.info("Model Loaded...")
        training_args = GRPOConfig(
            learning_rate = 5e-6,
            adam_beta1 = 0.9,
            adam_beta2 = 0.99,
            weight_decay = 0.1,
            warmup_ratio = 0.1,
            lr_scheduler_type = "cosine",
            optim = "paged_adamw_8bit",
            logging_steps = 1,
            generation_batch_size = 8,
            per_device_train_batch_size = 1,
            gradient_accumulation_steps = 1, # Increase to 4 for smoother training
            bf16=True,
            gradient_checkpointing=False,
            num_generations = 8, # Decrease if out of memory
            max_prompt_length = max_prompt_length,
            max_completion_length = max_seq_length,
            num_train_epochs = 1, # Set to 1 for a full training run
            save_steps = 10,
            max_steps = iteration+50,
            max_grad_norm = 0.1,
            report_to = "trackio", # Can use Weights & Biases
            run_name="roserun-1",
            output_dir = lora_id,
        )
        dataset = load_dataset("qwedsacf/competition_math", split="train")
        mapped = dataset.map(to_prompt_completion, remove_columns=dataset.column_names).shuffle()
        logger.info("Configured...")
        trainer = GRPOTrainer(
            model = model,
            processing_class = tokenizer,
            reward_funcs = [
                strict_format_reward_func,
                correctness_reward_func,
            ],
            args = training_args,
            train_dataset = mapped,
        )
        logger.info("Starting Training...")
        memory_stats()
        if loaded_from == base_model_id:
            trainer.train(resume_from_checkpoint=False)
        else:
            # resume from checkpoint needs changing max step
            trainer.train(resume_from_checkpoint=loaded_from)
        logger.info("Finished Training...")
        memory_stats()
        del model
        del tokenizer
        del trainer
        del dataset
        del mapped
        with torch.no_grad():
            torch.cuda.empty_cache()
        torch.cuda.empty_cache()
        gc.collect()
        logger.info("Cleaned up memory...")
        memory_stats()
    # ========================================================================
    # 3. STOP CRITERION TASK
    # ========================================================================
    @rl.as_stop_criterion(metric_name='MODEL_REWARD', threshold=.5, operator=GREATER_THAN_THRESHOLD, as_executable=False)
    async def check_reward(*args, **kwargs):
        def rewards_func(prompts, completions, ground_truth, **kwargs):
            rewards = []
            for prompt, completion, ground in zip(prompts, completions, ground_truth):
                c = get_answer(completion)
                g = get_answer(ground)
                reward = 1.0 if g == c else 0
                rewards.append(reward)
            return rewards
        #need to let the previous node get cleaned up
        time.sleep(10)
        batch_size=8
        iteration=1
        model, tokenizer, loaded_from, _ = load_llama_or_latest_checkpoint(
            base_model_id=base_model_id,
            lora_id=lora_id,
            dtype=torch.bfloat16,
        )
        dataset = load_dataset("qwedsacf/competition_math", split="train")
        mapped = dataset.map(to_prompt_completion, remove_columns=dataset.column_names)
        #model = AutoModelForCausalLM.from_pretrained(
        #    "meta-llama/Llama-3.1-8B-Instruct", dtype=torch.bfloat16, device_map="auto"
        #)
        #lora_model = PeftModel.from_pretrained(model, 
        #    "math_output",
        #     is_trainable=True
        #)
        #tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
        #tokenizer.pad_token = tokenizer.eos_token
        #lora_model.print_trainable_parameters()
        total_correct=0.0
        logger.info("Starting grading...")
        memory_stats()
        for step in range(iteration):
            shuffled_dataset = mapped.shuffle()
            #batch = random.sample(mapped, K)
            batch = shuffled_dataset.select(range(batch_size))
            #print("sampled")
            messages = [
                [{"role": "user", "content": ex["prompt"]}]
                for ex in batch
            ]
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                padding=True,
                return_tensors="pt",
            ).to(model.device)
            #print("tokenized")
            with torch.no_grad():
                outputs = model.generate(
                    inputs,
                    max_new_tokens=2048,
                    do_sample=True,      # sampling
                    top_p=0.9,
                    temperature=0.7,
                    pad_token_id = tokenizer.eos_token_id
                    # don't usually mix beam search + sampling;
                    # if you want beam search, drop top_p/temperature and set num_beams>1
                )
                
            #print("generated")
            texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            #for src, full in zip(batch, texts):
            #    print("PROMPT:", src["prompt"])
            #    print("OUTPUT:", full)
            #    print("-" * 80)
            rewards = rewards_func([prompt["prompt"] for prompt in batch], texts, [prompt["ground_truth"] for prompt in batch])
            #print(str(step)+"/"+str(iteration))
            del shuffled_dataset
            del batch
            del inputs
            del outputs
            del texts
            total_correct = total_correct + sum(rewards)/2
        logger.info("Finished grading...")
        memory_stats()
        del model
        del tokenizer
        del dataset
        del mapped
        with torch.no_grad():
            torch.cuda.empty_cache()
        torch.cuda.empty_cache()
        gc.collect()
        logger.info("Cleaned up memory...")
        memory_stats()
        return total_correct / (iteration*batch_size)

    # Run
    logger.info("Starting Reinforcement Learning with ROSE...")
    await rl.learn(skip_simulation_step=True,**kwargs)
    logger.info("Reinforcement Learning completed!")

try:
    engine = await ConcurrentExecutionBackend(ProcessPoolExecutor())
    asyncflow = await WorkflowEngine.create(engine)
    rl = SequentialReinforcementLearner(asyncflow)

    init_default_logger(logging.INFO)
    await run(rl, max_iter=1)
except Exception as e:
    print(f'Learner Failed with: {e}')
finally:
    await rl.shutdown()
    logging.getLogger().handlers.clear()

2025-11-25 23:32:34.038 │ INFO │ [root] │ Logger configured successfully - Console: INFO, File: disabled (N/A), Structured: disabled, Style: modern
2025-11-25 23:32:34.039 │ INFO │ [main] │ Starting Reinforcement Learning with ROSE...
Starting Sequential RL Learner
Starting Iteration-0
2025-11-25 23:32:34.040 │ INFO │ [workflow_manager] │ Submitting 1 tasks/blocks for execution
Found LoRA checkpoint at: math_lora/checkpoint-600
Loading base model: meta-llama/Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Applying LoRA adapter from checkpoint...
trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848
2025-11-25 23:33:40.178 │ INFO │ [main] │ Model Loaded...
2025-11-25 23:33:41.815 │ INFO │ [main] │ Configured...
2025-11-25 23:33:41.989 │ WARNING │ [other] │ Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2025-11-25 23:33:41.999 │ INFO │ [main] │ Starting Training...
memory allocated:  3086.140625
memory reserved:  3114.0


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


* Created new run: roserun-1


...

Applying LoRA adapter from checkpoint...
trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848
2025-11-25 23:44:50.903 │ INFO │ [main] │ Model Loaded...
2025-11-25 23:44:51.751 │ INFO │ [main] │ Configured...
2025-11-25 23:44:51.886 │ WARNING │ [other] │ Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2025-11-25 23:44:51.895 │ INFO │ [main] │ Starting Training...
memory allocated:  3086.140625
memory reserved:  3114.0


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


* Created new run: roserun-1


Step,Training Loss
651,-0.066100
652,-0.084400
653,-0.047900
654,-0.120100
655,0.099300
656,-0.056700
657,-0.058100
658,0.220800
659,0.034500
660,0.068100


* Run finished. Uploading logs to Trackio (please wait...)
2025-11-25 23:53:35.656 │ INFO │ [main] │ Finished Training...
memory allocated:  3104.8203125
memory reserved:  10920.0
2025-11-25 23:53:36.382 │ INFO │ [main] │ Cleaned up memory...
memory allocated:  1018.25
memory reserved:  3382.0
2025-11-25 23:53:36.404 │ INFO │ [workflow_manager] │ task.000003 is in DONE state
2025-11-25 23:53:36.404 │ INFO │ [main] │ Reinforcement Learning completed!
2025-11-25 23:53:36.405 │ INFO │ [workflow_manager] │ Initiating shutdown
2025-11-25 23:53:39.225 │ INFO │ [execution.backend(concurrent)] │ Concurrent execution backend shutdown complete
2025-11-25 23:53:39.226 │ INFO │ [workflow_manager] │ Shutdown completed for all components.
